# ECEi memmap for ecei_mc — train/val/test 80/10/10, balanced clear/disrupt

Build memory-mapped subsequence files from **ecei_mc** decimated data. Splits are **80% train, 10% val, 10% test** with **stratified** clear/disrupt (each type split 80/10/10). Clear shots have no disruptive labels; disrupt shots use `t_disruption` for the Twarn window. Subsequencing and save format follow `preprocessing_mmap.ipynb` / `preprocess_subseqs.py`.

**Output:** `output_dir/{train,val,test}/` each with RaggedMmap `X/`, `target/`, `weight/` and `labels.npy`. Requires `pip install mmap_ninja`.

In [ ]:
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import h5py
from tqdm import tqdm

try:
    from mmap_ninja import RaggedMmap
    HAS_MMAP_NINJA = True
except ImportError:
    HAS_MMAP_NINJA = False
    print("Install mmap_ninja: pip install mmap_ninja")

# ── Config (ecei_mc) ─────────────────────────────────────────────
BASE = Path("/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc")
CLEAR_DECIMATED = BASE / "clear_decimated"
DISRUPT_DECIMATED = BASE / "disrupt_decimated"
NORM_STATS_PATH = BASE / "norm_stats.npz"  # or compute below if missing
OUTPUT_DIR = BASE / "subseqs_mmap"
# t_disruption from DisruptCNN shot list (1 MHz: tstart, tdisrupt in ms; dt=0.001)
DISRUPT_SHOT_LIST = Path("disruptcnn/shots/d3d_disrupt_ecei.final.txt")

NSUB_RAW = 781_250
STRIDE_RAW = 481_090
DATA_STEP = 10
BASELINE_LEN = 40_000
TWARN_MS = 300
TWARN = TWARN_MS * 1000  # in raw samples at 1 MHz; in decimated space we use TWARN // DATA_STEP
RANDOM_SEED = 42
MMAP_BATCH_SIZE = 512
N_WORKERS = 5

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.8, 0.1, 0.1
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-6

print(f"Clear decimated: {CLEAR_DECIMATED}")
print(f"Disrupt decimated: {DISRUPT_DECIMATED}")
print(f"Output: {OUTPUT_DIR}")
print(f"mmap_ninja: {HAS_MMAP_NINJA}")

## 1. List shots and stratified 80/10/10 split

List clear and disrupt shots from decimated dirs. Split each group 80/10/10 (train/val/test) with a fixed seed so splits are reproducible. Optionally read `t_disruption` from meta.csv in disrupt_decimated.

In [ ]:
def list_shots(decimated_root: Path) -> list[int]:
    if not decimated_root.exists():
        return []
    return sorted(int(p.stem) for p in decimated_root.glob("*.h5") if p.stem.isdigit())


def stratified_split(shot_ids: list[int], train_frac: float, val_frac: float, seed: int):
    n = len(shot_ids)
    if n == 0:
        return [], [], []
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    n_test = n - n_train - n_val
    train_inds = perm[:n_train]
    val_inds = perm[n_train : n_train + n_val]
    test_inds = perm[n_train + n_val :]
    train_shots = [shot_ids[i] for i in train_inds]
    val_shots = [shot_ids[i] for i in val_inds]
    test_shots = [shot_ids[i] for i in test_inds]
    return train_shots, val_shots, test_shots


clear_shots = list_shots(CLEAR_DECIMATED)
disrupt_shots = list_shots(DISRUPT_DECIMATED)
print(f"Clear:   {len(clear_shots)} shots")
print(f"Disrupt: {len(disrupt_shots)} shots")

clear_train, clear_val, clear_test = stratified_split(clear_shots, TRAIN_FRAC, VAL_FRAC, RANDOM_SEED)
disrupt_train, disrupt_val, disrupt_test = stratified_split(disrupt_shots, TRAIN_FRAC, VAL_FRAC, RANDOM_SEED)
print(f"Clear   -> train {len(clear_train)}, val {len(clear_val)}, test {len(clear_test)}")
print(f"Disrupt -> train {len(disrupt_train)}, val {len(disrupt_val)}, test {len(disrupt_test)}")

# t_disruption for disrupt shots from d3d_disrupt_ecei.final.txt (1 MHz: tstart, tdisrupt in ms; dt=0.001)
# Columns: Shot=0, tstart=2, tlast=3, dt=4, tdisrupt=8. Decimated index = (tdisrupt - tstart) / dt / DATA_STEP
t_disruption_by_shot = {}
if DISRUPT_SHOT_LIST.exists():
    data = np.loadtxt(DISRUPT_SHOT_LIST, skiprows=1)
    if data.ndim == 1:
        data = data[np.newaxis, :]
    shot_col, tstart_col, tdisrupt_col = 0, 2, 8
    for i in range(len(data)):
        shot = int(data[i, shot_col])
        tstart_ms = float(data[i, tstart_col])
        tdisrupt_ms = float(data[i, tdisrupt_col])
        if tdisrupt_ms < 0:
            continue
        # 1 MHz: raw samples = (tdisrupt - tstart) / 0.001; decimated = raw / DATA_STEP
        t_disruption_by_shot[shot] = int((tdisrupt_ms - tstart_ms) * 1000 / DATA_STEP)
    print(f"t_disruption loaded for {len(t_disruption_by_shot)} disrupt shots from {DISRUPT_SHOT_LIST}")
else:
    print(f"Shot list not found: {DISRUPT_SHOT_LIST}; disrupt shots will use full file length")

## 2. Build shot metadata and subsequence index

Build unified shot list (clear + disrupt) with split labels, start/stop/disrupt indices in **decimated** sample space. Clear shots: segment = full file, no disruption (disrupt_idx past end). Disrupt shots: segment up to t_disruption (or file length if missing); disrupt_idx = start of Twarn window. Then build subsequence index (same logic as preprocess_subseqs).

In [ ]:
q = DATA_STEP  # decimated: 1 sample = 10 raw
baseline_dec = BASELINE_LEN // q
twarn_dec = TWARN // q

def build_shot_arrays():
    shots_list, splits_list = [], []
    start_list, stop_list, disrupt_list, pos_end_list = [], [], [], []
    roots_list, step_list = [], []
    failed_shots = []  # (shot, "clear"|"disrupt") for files that raise OSError/IOError

    def add_shots(shot_ids: list, split_name: str, root: Path, is_clear: bool):
        group = "clear" if is_clear else "disrupt"
        for shot in shot_ids:
            path = root / f"{shot}.h5"
            if not path.exists():
                continue
            try:
                with h5py.File(path, "r") as f:
                    T = f["LFS"].shape[-1]
            except (OSError, IOError):
                failed_shots.append((shot, group))
                continue
            start_list.append(baseline_dec)
            stop_list.append(T)
            if is_clear:
                disrupt_list.append(T + 1)
                pos_end_list.append(T)
            else:
                t_dis = t_disruption_by_shot.get(shot, T)
                t_dis = min(t_dis, T)
                dis_idx = max(0, t_dis - twarn_dec)
                disrupt_list.append(dis_idx)
                pos_end_list.append(t_dis)
            shots_list.append(shot)
            splits_list.append(split_name)
            roots_list.append(root)
            step_list.append(1)

    for sid, name in [(clear_train, "train"), (clear_val, "val"), (clear_test, "test")]:
        add_shots(sid, name, CLEAR_DECIMATED, is_clear=True)
    for sid, name in [(disrupt_train, "train"), (disrupt_val, "val"), (disrupt_test, "test")]:
        add_shots(sid, name, DISRUPT_DECIMATED, is_clear=False)

    return (
        np.array(shots_list, dtype=int),
        np.array(splits_list, dtype=object),
        np.array(start_list, dtype=np.int64),
        np.array(stop_list, dtype=np.int64),
        np.array(disrupt_list, dtype=np.int64),
        np.array(pos_end_list, dtype=np.int64),
        roots_list,
        step_list,
        failed_shots,
    )


result = build_shot_arrays()
shots_arr, splits_arr, start_idx, stop_idx, disrupt_idx, positive_end_idx, shot_data_root, shot_step, failed_shots = result
n_shots = len(shots_arr)
print(f"Total shots: {n_shots} (train {np.sum(splits_arr == 'train')}, val {np.sum(splits_arr == 'val')}, test {np.sum(splits_arr == 'test')})")
if failed_shots:
    clear_fail = sorted(s for s, g in failed_shots if g == "clear")
    disrupt_fail = sorted(s for s, g in failed_shots if g == "disrupt")
    print(f"Skipped {len(failed_shots)} erroneous file(s) (OSError/IOError): clear {clear_fail}, disrupt {disrupt_fail}")

In [ ]:
def build_subsequence_index(
    n_shots, start_idx, stop_idx, disrupt_idx, positive_end_idx, nsub, stride, q
):
    data_nsub = nsub // q
    data_stride = stride // q
    shot_idx, starts, stops, d_local, e_local = [], [], [], [], []
    for s in range(n_shots):
        a, b = int(start_idx[s]), int(stop_idx[s])
        if b - a < data_nsub:
            continue
        d = int(disrupt_idx[s])
        e_abs = int(positive_end_idx[s])
        pos = a
        while pos + data_nsub <= b:
            shot_idx.append(s)
            starts.append(pos)
            stops.append(pos + data_nsub)
            if d <= pos:
                d_local.append(0)
                e_local.append(min(pos + data_nsub, e_abs) - pos)
            elif d >= pos + data_nsub:
                d_local.append(-1)
                e_local.append(0)
            else:
                d_local.append(d - pos)
                e_local.append(min(pos + data_nsub, e_abs) - pos)
            pos += data_stride
        last_start = b - data_nsub
        if last_start > (pos - data_stride):
            shot_idx.append(s)
            starts.append(last_start)
            stops.append(b)
            if d <= last_start:
                d_local.append(0)
                e_local.append(min(b, e_abs) - last_start)
            elif d >= b:
                d_local.append(-1)
                e_local.append(0)
            else:
                d_local.append(d - last_start)
                e_local.append(min(b, e_abs) - last_start)
    return (
        np.array(shot_idx, dtype=int),
        np.array(starts, dtype=int),
        np.array(stops, dtype=int),
        np.array(d_local, dtype=int),
        np.array(e_local, dtype=int),
    )


stride_raw = STRIDE_RAW
seq_shot_idx, seq_start, seq_stop, seq_disrupt_local, seq_positive_end_local = build_subsequence_index(
    n_shots, start_idx, stop_idx, disrupt_idx, positive_end_idx,
    NSUB_RAW, stride_raw, q,
)
n_seq = len(seq_shot_idx)
T_sub = NSUB_RAW // q
step_in = 1
print(f"Subsequences: {n_seq}, T_sub (decimated length per window): {T_sub}")

In [ ]:
def compute_labels_and_weights(T, dl, el, step, ignore_twarn=False, pos_weight=1.0, neg_weight=1.0):
    target = np.zeros(T, dtype=np.float32)
    weight = np.full(T, neg_weight, dtype=np.float32)
    if dl >= 0:
        d = min((dl + 1) // step, T)
        if ignore_twarn:
            weight[d:] = 0.0
        else:
            e = min((el + step - 1) // step, T)
            e = max(d, e)
            target[d:e] = 1.0
            weight[d:e] = pos_weight
            if e < T:
                weight[e:] = 0.0
    return target, weight


# Class weights from full train subsequences
train_mask = splits_arr[seq_shot_idx] == "train"
train_seq_idx = np.where(train_mask)[0]
total_pos = total_neg = 0
for i in train_seq_idx:
    dl, el = int(seq_disrupt_local[i]), int(seq_positive_end_local[i])
    if dl < 0:
        total_neg += T_sub
    else:
        d = (dl + 1) // step_in
        e = min((el + step_in - 1) // step_in, T_sub)
        e = max(d, e)
        total_pos += e - d
        total_neg += d + (T_sub - e)
total = total_pos + total_neg
pos_weight = float(0.5 * total / total_pos) if total_pos > 0 else 1.0
neg_weight = float(0.5 * total / total_neg) if total_neg > 0 else 1.0
print(f"pos_weight: {pos_weight:.4f}, neg_weight: {neg_weight:.4f}")

## 3. Normalization stats

Load `norm_stats.npz` if present; otherwise compute from all decimated shots in parallel (N_WORKERS). Run this first; memmap write (section 4) uses these stats.

In [ ]:
CHANNELS = 20 * 8
if NORM_STATS_PATH.exists():
    nz = np.load(NORM_STATS_PATH)
    norm_mean = nz["mean"].astype(np.float32)
    norm_std = nz["std"].astype(np.float32)
    print(f"Loaded norm stats from {NORM_STATS_PATH}")
else:
    def _load_shot_stats(path):
        try:
            with h5py.File(path, "r") as f:
                X = np.asarray(f["LFS"][:], dtype=np.float64)
        except (OSError, IOError, KeyError):
            return None
        flat = X.reshape(CHANNELS, -1)
        n = flat.shape[1]
        return n, flat.sum(axis=1), (flat ** 2).sum(axis=1)

    all_paths = list(CLEAR_DECIMATED.glob("*.h5")) + list(DISRUPT_DECIMATED.glob("*.h5"))
    all_paths = [p for p in all_paths if p.stem.isdigit()]
    n_tot = 0
    sum_x = np.zeros(CHANNELS, dtype=np.float64)
    sum_sq = np.zeros(CHANNELS, dtype=np.float64)
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        for res in tqdm(as_completed([pool.submit(_load_shot_stats, p) for p in all_paths]), total=len(all_paths), desc="Norm stats"):
            r = res.result()
            if r is None:
                continue
            n, sx, sq = r
            n_tot += n
            sum_x += sx
            sum_sq += sq
    norm_mean = (sum_x / n_tot).astype(np.float32).reshape(20, 8)
    var = (sum_sq / n_tot) - (sum_x / n_tot) ** 2
    norm_std = np.sqrt(np.maximum(var, 0)).astype(np.float32).reshape(20, 8)
    np.savez(NORM_STATS_PATH, mean=norm_mean, std=norm_std)
    print(f"Computed and saved norm stats to {NORM_STATS_PATH}")
if norm_mean.ndim == 1:
    norm_mean = norm_mean.reshape(20, 8)
    norm_std = norm_std.reshape(20, 8)
assert norm_mean.shape == (20, 8) and norm_std.shape == (20, 8)

## 4. Write train / val / test memmaps

Requires norm stats (section 3) and shot/subsequence metadata (section 2). Reads decimated H5, normalizes, builds (X, target, weight) per subsequence, streams to RaggedMmap per split. **Wipes each split dir before writing** so re-runs never append.

In [ ]:
import shutil
import time
from collections import defaultdict

CHUNK_SIZE = 128
FLUSH_BATCH_SIZE = 512

if not HAS_MMAP_NINJA:
    raise RuntimeError("mmap_ninja required; pip install mmap_ninja")

# ── Load norm stats ───────────────────────────────────────────────
try:
    _ = norm_mean; _ = norm_std
    print("Using norm_mean / norm_std from kernel (section 3).")
except NameError:
    print(f"Loading norm stats from {NORM_STATS_PATH} ...")
    nz = np.load(NORM_STATS_PATH)
    norm_mean = nz["mean"].astype(np.float32)
    norm_std  = nz["std"].astype(np.float32)
    if norm_mean.ndim == 1:
        norm_mean = norm_mean.reshape(20, 8)
        norm_std  = norm_std.reshape(20, 8)
    print(f"  norm_mean.shape={norm_mean.shape}, norm_std.shape={norm_std.shape}")

# ── Precompute split indices and expected sizes ────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
split_names = ["train", "val", "test"]
split_indices = {name: np.where(splits_arr[seq_shot_idx] == name)[0] for name in split_names}
meta_out = {"nsub": NSUB_RAW, "stride": stride_raw, "data_step": DATA_STEP, "T_sub": T_sub,
            "pos_weight": pos_weight, "neg_weight": neg_weight, "twarn_ms": TWARN_MS}

bytes_per_seg_X = 20 * 8 * T_sub * 4  # float32
bytes_per_seg_t = T_sub * 4
bytes_per_seg_w = T_sub * 4
bytes_per_seg   = bytes_per_seg_X + bytes_per_seg_t + bytes_per_seg_w

print(f"CHUNK_SIZE={CHUNK_SIZE}, FLUSH_BATCH_SIZE={FLUSH_BATCH_SIZE}")
print(f"Per-segment size: X={bytes_per_seg_X/1e6:.2f} MB, target+weight={2*bytes_per_seg_t/1e6:.2f} MB, total={bytes_per_seg/1e6:.2f} MB")
print()
for name in split_names:
    n = len(split_indices[name])
    gb = n * bytes_per_seg / (1024**3)
    print(f"  {name:5s}: {n:>8,} segments  ->  expected {gb:>7.1f} GB on disk")
n_total = sum(len(split_indices[s]) for s in split_names)
total_gb = n_total * bytes_per_seg / (1024**3)
print(f"  {'total':5s}: {n_total:>8,} segments  ->  expected {total_gb:>7.1f} GB on disk")

# ── prepare_chunk: read H5, normalize, build (X, target, weight) ──
def prepare_chunk(indices_chunk):
    items = []
    for idx in indices_chunk:
        s = int(seq_shot_idx[idx])
        shot = int(shots_arr[s])
        root = shot_data_root[s]
        start, stop = int(seq_start[idx]), int(seq_stop[idx])
        dl, el = int(seq_disrupt_local[idx]), int(seq_positive_end_local[idx])
        items.append((root, shot, start, stop, dl, el))
    groups = defaultdict(list)
    for orig_idx, (root, shot, start, stop, dl, el) in enumerate(items):
        groups[(root, shot)].append((start, stop, dl, el, orig_idx))
    out = [None] * len(indices_chunk)
    for (root, shot), group in groups.items():
        try:
            with h5py.File(root / f"{shot}.h5", "r") as f:
                dset = f["LFS"]
                for start, stop, dl, el, orig_idx in group:
                    X = np.asarray(dset[..., start:stop], dtype=np.float32)
                    X = (X - norm_mean[..., np.newaxis]) / norm_std[..., np.newaxis]
                    target, weight = compute_labels_and_weights(
                        T_sub, dl, el, step_in, ignore_twarn=False,
                        pos_weight=pos_weight, neg_weight=neg_weight)
                    out[orig_idx] = (np.ascontiguousarray(X), target,
                                     weight.astype(np.float32),
                                     1 if dl >= 0 else 0)
        except (OSError, IOError, KeyError):
            pass
    return out

# ── Write each split ──────────────────────────────────────────────
for split_name in split_names:
    indices = split_indices[split_name]
    meta_out[f"n_{split_name}"] = len(indices)
    if len(indices) == 0:
        print(f"\n[{split_name}] 0 segments, skipping.")
        continue

    subdir = OUTPUT_DIR / split_name
    # CRITICAL: wipe previous output so re-runs don't append
    if subdir.exists():
        shutil.rmtree(subdir)
    subdir.mkdir(parents=True, exist_ok=True)

    chunks = [indices[i : i + CHUNK_SIZE] for i in range(0, len(indices), CHUNK_SIZE)]
    n_chunks = len(chunks)
    print(f"\n[{split_name}] {len(indices):,} segments, {n_chunks} chunks -> {subdir}")

    X_buf, t_buf, w_buf = [], [], []
    labels_split = []
    n_written = 0
    n_skipped = 0
    mmap_initialized = False
    x_mmap = t_mmap = w_mmap = None

    def flush_buf(n):
        """Write first n items from buffers to RaggedMmap, then discard them."""
        global mmap_initialized, x_mmap, t_mmap, w_mmap
        nonlocal n_written
        if n <= 0 or len(X_buf) < n:
            return
        xs, ts, ws = X_buf[:n], t_buf[:n], w_buf[:n]
        if not mmap_initialized:
            RaggedMmap.from_lists(str(subdir / "X"), xs)
            RaggedMmap.from_lists(str(subdir / "target"), ts)
            RaggedMmap.from_lists(str(subdir / "weight"), ws)
            x_mmap = RaggedMmap(str(subdir / "X"))
            t_mmap = RaggedMmap(str(subdir / "target"))
            w_mmap = RaggedMmap(str(subdir / "weight"))
            mmap_initialized = True
        else:
            x_mmap.extend(xs)
            t_mmap.extend(ts)
            w_mmap.extend(ws)
        n_written += n
        del X_buf[:n], t_buf[:n], w_buf[:n]

    split_t0 = time.perf_counter()
    log_interval = max(1, n_chunks // 20)  # ~20 progress prints per split

    for c, chunk in enumerate(chunks):
        chunk_t0 = time.perf_counter()
        results = prepare_chunk(chunk)
        for r in results:
            if r is None:
                n_skipped += 1
                continue
            X_buf.append(r[0])
            t_buf.append(r[1])
            w_buf.append(r[2])
            labels_split.append(r[3])
        while len(X_buf) >= FLUSH_BATCH_SIZE:
            flush_buf(FLUSH_BATCH_SIZE)
        if (c + 1) % log_interval == 0 or c == 0 or c == n_chunks - 1:
            elapsed = time.perf_counter() - split_t0
            rate = n_written / elapsed if elapsed > 0 else 0
            pct = 100 * (c + 1) / n_chunks
            print(f"  [{split_name}] chunk {c+1:>6d}/{n_chunks} ({pct:5.1f}%)  "
                  f"written={n_written:>8,}  skipped={n_skipped}  "
                  f"elapsed={elapsed:.0f}s  ({rate:.0f} seg/s)")

    # flush remainder
    flush_buf(len(X_buf))
    # close mmap handles so OS can reclaim resources
    del x_mmap, t_mmap, w_mmap
    x_mmap = t_mmap = w_mmap = None

    split_elapsed = time.perf_counter() - split_t0
    np.save(subdir / "labels.npy", np.array(labels_split, dtype=np.int64))
    rate = n_written / split_elapsed if split_elapsed > 0 else 0
    actual_gb = n_written * bytes_per_seg / (1024**3)
    print(f"  [{split_name}] DONE: {n_written:,} written, {n_skipped} skipped  |  "
          f"{split_elapsed:.0f}s ({rate:.0f} seg/s)  |  ~{actual_gb:.1f} GB")

with open(OUTPUT_DIR / "meta.json", "w") as f:
    json.dump(meta_out, f, indent=2)
print(f"\nDone. Saved to {OUTPUT_DIR}")
print(json.dumps(meta_out, indent=2))

## 5. Verify

In [ ]:
x_m = RaggedMmap(str(OUTPUT_DIR / "train" / "X"))
t_m = RaggedMmap(str(OUTPUT_DIR / "train" / "target"))
w_m = RaggedMmap(str(OUTPUT_DIR / "train" / "weight"))
labels = np.load(OUTPUT_DIR / "train" / "labels.npy")
print(f"Train: {len(x_m)} subsequences")
print(f"  X[0].shape: {np.asarray(x_m[0]).shape}  (expect (20, 8, {T_sub}))")
print(f"  target[0].shape: {t_m[0].shape}")
print(f"  labels (has_disrupt): {labels[:10]} ...")

---

In [ ]:
# Done. Run section 3 (norm stats), then section 4 (memmaps); verify in section 5.